# Probability & Statistics — Prerequisite Review

**Who is this for?** Students entering the ML course who already know basic probability and statistics but want a quick refresher and self-check.

**Estimated time:** 30-45 minutes.

**What this covers:**
- Descriptive statistics (mean, median, variance, standard deviation)
- Probability basics (events, sample space, independence)
- Conditional probability and Bayes' theorem
- Common distributions (uniform, normal, Bernoulli, binomial)
- Sampling and the law of large numbers
- Visualizing distributions

If any section feels completely unfamiliar (not just rusty), stop and review the references at the bottom before continuing the course.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 3.5)
plt.rcParams["axes.grid"] = True

---
## 1. Descriptive Statistics

Given a dataset $x_1, x_2, \dots, x_n$:

| Statistic | Formula |
|---|---|
| **Mean** | $\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$ |
| **Median** | Middle value when sorted |
| **Variance** | $\sigma^2 = \frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2$ |
| **Std. deviation** | $\sigma = \sqrt{\sigma^2}$ |

Note: NumPy's `np.var` and `np.std` use the population formula by default ($n$). For the sample estimator ($n-1$), pass `ddof=1`.

In [ ]:
# Example: descriptive stats on a small dataset
data = np.array([4, 8, 6, 5, 3, 8, 9, 2, 7, 6])

print(f"Mean:    {np.mean(data):.2f}")
print(f"Median:  {np.median(data):.2f}")
print(f"Var:     {np.var(data):.2f}  (population)")
print(f"Var:     {np.var(data, ddof=1):.2f}  (sample)")
print(f"Std:     {np.std(data):.2f}")

### Exercise 1.1

Generate 1000 samples from a uniform distribution on $[0, 10]$ using `np.random.uniform`. Compute the mean, median, variance, and standard deviation. Compare to the theoretical values ($\mu = 5$, $\sigma^2 = 100/12 \approx 8.33$).

In [ ]:
# YOUR CODE HERE

### Exercise 1.2

Create a dataset where the mean and median differ significantly (e.g., a skewed distribution). Print both values and explain in a comment why they differ.

In [ ]:
# YOUR CODE HERE

---
## 2. Probability Basics

**Sample space** $\Omega$: the set of all possible outcomes.  
**Event** $A \subseteq \Omega$: a subset of outcomes.

$$P(A) = \frac{|A|}{|\Omega|} \quad \text{(equally likely outcomes)}$$

Key rules:
- $0 \le P(A) \le 1$
- $P(A \cup B) = P(A) + P(B) - P(A \cap B)$
- **Independence:** $A$ and $B$ are independent iff $P(A \cap B) = P(A) \cdot P(B)$

In [ ]:
# Example: simulate two fair dice — verify independence
n_rolls = 100_000
die1 = np.random.randint(1, 7, size=n_rolls)
die2 = np.random.randint(1, 7, size=n_rolls)

# Event A: die1 is even.  Event B: die2 >= 5
p_a = np.mean(die1 % 2 == 0)
p_b = np.mean(die2 >= 5)
p_ab = np.mean((die1 % 2 == 0) & (die2 >= 5))

print(f"P(A)={p_a:.4f}  P(B)={p_b:.4f}  P(A)P(B)={p_a*p_b:.4f}  P(A∩B)={p_ab:.4f}")
print(f"Independent? {np.isclose(p_ab, p_a * p_b, atol=0.01)}")

### Exercise 2.1

Simulate 100,000 rolls of a single fair die. Estimate $P(\text{roll} \le 2)$ and compare to the theoretical $1/3$.

In [ ]:
# YOUR CODE HERE

---
## 3. Conditional Probability & Bayes' Theorem

**Conditional probability:**

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

**Bayes' theorem:**

$$P(A \mid B) = \frac{P(B \mid A) \, P(A)}{P(B)}$$

**Intuition:** Bayes' theorem lets you update your belief about $A$ after observing evidence $B$. The prior $P(A)$ becomes the posterior $P(A|B)$ by accounting for how likely the evidence is under $A$ vs. overall.

In [ ]:
# Classic example: disease testing
# Disease prevalence: 1%
# Test sensitivity (true positive rate): 95%
# Test specificity (true negative rate): 90%

p_disease = 0.01
sensitivity = 0.95  # P(positive | disease)
specificity = 0.90  # P(negative | no disease)

# P(positive) by total probability
p_positive = sensitivity * p_disease + (1 - specificity) * (1 - p_disease)

# P(disease | positive) via Bayes
p_disease_given_pos = (sensitivity * p_disease) / p_positive

print(f"P(disease | positive test) = {p_disease_given_pos:.4f}")
print(f"Despite a 95% sensitive test, only ~{p_disease_given_pos:.1%} of positives actually have the disease.")
print(f"This is because the disease is rare (1%), so false positives dominate.")

In [ ]:
# Simulation to verify
n_people = 1_000_000
has_disease = np.random.rand(n_people) < p_disease

# Generate test results
test_positive = np.where(
    has_disease,
    np.random.rand(n_people) < sensitivity,    # true positive
    np.random.rand(n_people) < (1 - specificity)  # false positive
)

sim_p = has_disease[test_positive].mean()
print(f"Simulated P(disease | positive) = {sim_p:.4f}  (analytical: {p_disease_given_pos:.4f})")

### Exercise 3.1

A factory has two machines. Machine A produces 60% of items, Machine B produces 40%. Defect rates: A = 2%, B = 5%. An item is found defective. What is the probability it came from Machine B? Compute analytically with Bayes' theorem, then verify with a simulation.

In [ ]:
# YOUR CODE HERE

---
## 4. Common Distributions

| Distribution | Support | Parameters | Mean | Variance |
|---|---|---|---|---|
| **Bernoulli** | $\{0,1\}$ | $p$ | $p$ | $p(1-p)$ |
| **Binomial** | $\{0,\dots,n\}$ | $n, p$ | $np$ | $np(1-p)$ |
| **Uniform** | $[a, b]$ | $a, b$ | $\frac{a+b}{2}$ | $\frac{(b-a)^2}{12}$ |
| **Normal** | $(-\infty, \infty)$ | $\mu, \sigma^2$ | $\mu$ | $\sigma^2$ |

The **normal (Gaussian)** PDF:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

# Bernoulli
p = 0.3
samples = np.random.binomial(1, p, size=10000)
axes[0].bar([0, 1], [np.mean(samples == 0), np.mean(samples == 1)], color="steelblue")
axes[0].set_title(f"Bernoulli(p={p})")
axes[0].set_xticks([0, 1])

# Binomial
n, p = 20, 0.4
samples = np.random.binomial(n, p, size=10000)
axes[1].hist(samples, bins=np.arange(-0.5, n + 1.5), density=True, color="steelblue", edgecolor="white")
axes[1].set_title(f"Binomial(n={n}, p={p})")

# Uniform
samples = np.random.uniform(0, 1, size=10000)
axes[2].hist(samples, bins=30, density=True, color="steelblue", edgecolor="white")
axes[2].set_title("Uniform(0, 1)")

# Normal
samples = np.random.normal(0, 1, size=10000)
axes[3].hist(samples, bins=50, density=True, color="steelblue", edgecolor="white", alpha=0.7)
x = np.linspace(-4, 4, 200)
axes[3].plot(x, 1/np.sqrt(2*np.pi) * np.exp(-x**2/2), "r-", lw=2, label="PDF")
axes[3].legend()
axes[3].set_title("Normal(0, 1)")

plt.tight_layout()
plt.show()

### Exercise 4.1

Generate 10,000 samples from $\text{Binomial}(n=50, p=0.5)$. Plot a histogram and overlay a normal distribution $\mathcal{N}(np, np(1-p))$ to visually confirm the central limit theorem approximation.

In [ ]:
# YOUR CODE HERE

### Exercise 4.2

Sample 5,000 values from $\mathcal{N}(\mu=100, \sigma=15)$. Compute the fraction of samples within 1, 2, and 3 standard deviations of the mean. Compare to the 68-95-99.7 rule.

In [ ]:
# YOUR CODE HERE

---
## 5. Sampling and the Law of Large Numbers

The **Law of Large Numbers (LLN):** As $n \to \infty$, the sample mean $\bar{x}_n$ converges to the true mean $\mu$.

$$\bar{x}_n = \frac{1}{n}\sum_{i=1}^n x_i \xrightarrow{n \to \infty} \mu$$

This is why Monte Carlo methods work: simulate enough samples and the empirical average approximates the expectation.

In [ ]:
# Demo: watch the running mean converge
samples = np.random.exponential(scale=2.0, size=10000)  # true mean = 2.0
running_mean = np.cumsum(samples) / np.arange(1, len(samples) + 1)

plt.figure(figsize=(8, 3.5))
plt.plot(running_mean, color="steelblue", lw=1)
plt.axhline(2.0, color="red", ls="--", lw=1.5, label="True mean = 2.0")
plt.xlabel("Number of samples")
plt.ylabel("Running mean")
plt.title("Law of Large Numbers — Exponential(scale=2)")
plt.xscale("log")
plt.legend()
plt.tight_layout()
plt.show()

### Exercise 5.1

Estimate $\pi$ using Monte Carlo: sample points uniformly in the unit square $[0,1]^2$, count the fraction inside the quarter circle ($x^2 + y^2 \le 1$), multiply by 4. Plot the running estimate as a function of the number of samples.

In [ ]:
# YOUR CODE HERE

---
## 6. Visualizing Distributions

Two main tools:
- **Histogram** (`plt.hist`): bin counts, use `density=True` to normalize to a probability density.
- **Kernel Density Estimate (KDE)**: smooth estimate of the PDF. NumPy does not have a built-in KDE, but you can approximate with a fine histogram or use `scipy.stats.gaussian_kde`.

Key `plt.hist` parameters: `bins`, `density`, `alpha`, `edgecolor`.

In [ ]:
# Example: comparing two distributions visually
data_a = np.random.normal(5, 1.5, size=5000)
data_b = np.random.normal(7, 2.0, size=5000)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

# Overlapping histograms
axes[0].hist(data_a, bins=40, density=True, alpha=0.6, label="A ~ N(5, 1.5)", color="steelblue")
axes[0].hist(data_b, bins=40, density=True, alpha=0.6, label="B ~ N(7, 2.0)", color="coral")
axes[0].legend()
axes[0].set_title("Overlapping Histograms")

# Histogram + smoothed density (using fine-bin histogram as poor-man's KDE)
counts, bin_edges = np.histogram(data_a, bins=100, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
axes[1].hist(data_a, bins=40, density=True, alpha=0.4, color="steelblue", edgecolor="white")
axes[1].plot(bin_centers, counts, "r-", lw=2, label="Smoothed density")
axes[1].legend()
axes[1].set_title("Histogram + Density Curve")

plt.tight_layout()
plt.show()

### Exercise 6.1

Generate samples from three distributions: $\mathcal{N}(0,1)$, $\text{Uniform}(-3,3)$, and $\text{Exponential}(\lambda=1)$. Plot all three as overlapping histograms (use `density=True` and `alpha=0.5`) in a single figure with a legend. What differences in shape do you observe?

In [ ]:
# YOUR CODE HERE

---
## Self-Assessment

You should be able to answer these without looking anything up:

1. **What is the difference between population variance and sample variance?** When do you divide by $n$ vs. $n-1$, and why?

2. **A medical test has 99% sensitivity and 95% specificity. The disease prevalence is 0.1%.** If a patient tests positive, what is the approximate probability they actually have the disease? (Work it out with Bayes' theorem.)

3. **Why does the normal distribution appear so often in practice?** (Hint: Central Limit Theorem.)

4. **If you flip a fair coin 10,000 times, how close do you expect the fraction of heads to be to 0.5?** What principle guarantees this convergence?

If you struggled with more than one of these, review the resources below before starting the course.

---
## References

1. **Khan Academy — Statistics and Probability**  
   [https://www.khanacademy.org/math/statistics-probability](https://www.khanacademy.org/math/statistics-probability)  
   Comprehensive, free video-based review of all topics covered here.

2. **Think Stats (2nd ed.) — Allen Downey** (free online)  
   [https://greenteapress.com/thinkstats2/](https://greenteapress.com/thinkstats2/)  
   Python-based intro to probability and statistics. Good for the programming-oriented learner.

3. **3Blue1Brown — Probability** (YouTube)  
   [https://www.youtube.com/playlist?list=PLZHQObOWTQDOjmo3Y6ADm0ScWAlEXf-fp](https://www.youtube.com/playlist?list=PLZHQObOWTQDOjmo3Y6ADm0ScWAlEXf-fp)  
   Excellent visual intuition for Bayes' theorem and distributions.